# AI News Aggregator

## Project steps
   - ✅ Pahse 1 : Collect news from multiple sources 
   - ✅ Pahse 2 : Summarize articles using AI
   - ✅ Pahse 3 : Detect and remove duplicate news
   - ✅ Pahse 4 : Categorize articles by topic
   - ✅ Pahse 5 : Use an AI agent to rank news based on relevance and importance
   - ✅ Pahse 6 : Store the processed data in an SQL database
   - ✅ Pahse 7 : Generate and send a newsletter
   - ✅ Phase 8 — FastAPI
   - ✅ Phase 9 — Celery + Redis
   - ✅ Phase 10 — Dockerize
   - ✅ Phase 11 — Vector Search
   - ✅ Phase 12 — AI Chat Over Your News
   - ✅ Phase 13 — Frontend
   - ✅ Phase 13 — Deployment


#### create project structure ( later I will add more directories)
``` python 
# 1. Create the project folder and all subdirectories
mkdir -p app/{agents,collectors,llm,models,database,services,api,workers,scheduler}

# 2. Create all the empty Python files
touch app/agents/news_agent.py \
      app/collectors/{youtube,rss,blogs,twitter}.py \
      app/llm/provider.py \
      app/models/article.py \
      app/database/database.py \
      app/services/{summarizer,classifier,newsletter}.py \
      app/api/routes.py \
      app/workers/celery_worker.py \
      app/scheduler/jobs.py \
      app/{config,main}.py
      app/promts/summarizer.py

# 3. Create root files
touch   {docker-compose.yml,Dockerfile,requirements.txt,.env,README.md}
```
-----------------------------------------

##  ✅ Phase 1 — Collect News

Build a system that can collect news from multiple sources and return it in a single standardized format.

```
RSS Feed
        \
YouTube -----> Collectors -----> Standard Article Object
        /
Blogs
```

At the end of Phase 1, running the application should produce something like:

```python
[
    {
        "title": "...",
        "content": "...",
        "url": "...",
        "published_at": "...",
        "source": "OpenAI Blog"
    },
    ...
]
```
### ⭐ Steps :
1. I will use these 3 sources:

| Source       | RSS                                                                              |
| ------------ | -------------------------------------------------------------------------------- |
| OpenAI       | [https://openai.com/news/rss.xml](https://openai.com/news/rss.xml)               |
| Anthropic    | [https://www.anthropic.com/news/rss.xml](https://www.anthropic.com/news/rss.xml) |
| Hugging Face | [https://huggingface.co/blog/feed.xml](https://huggingface.co/blog/feed.xml)     |

2. Normalize the sources
Different sources expose different fields.we should normalize them into one structure.
The model should contain:
- title
- summary
- url
- published_at
- source

Every collector must return this exact model, regardless of where the article came from.

3. Create Collectors
- blogs
- rss
- youtube
 
and each collector should return the same model.
```
Fetch RSS
↓
Read entries
↓
Convert entries to Article objects
↓
Return a list of articles
```

4. Create a Collector Interface
Every collector should expose a collect() method.

5. Orchestrator
main.py should not know how RSS works. Instead, it simply coordinates collectors:
```
OpenAI Collector
        │
Anthropic Collector
        │
HF Collector
        │
        ▼
Merge results
        ▼
Print
``` 
This separation is important. main.py orchestrates; collectors implement source-specific logic.

### Folder Structure After Phase 1
```
ai-news-aggregator/
│
├── app/
│   ├── collectors/
│   │   ├── rss.py
│   │   ├── blogs.py
│   │   ├── twitter.py
│   │   └── youtube.py
│   │
│   ├── models/
│   │   └── article.py
│   │
│   ├── config.py
│   └── main.py
│
├── requirements.txt
├── .env
└── README.md
```
----------------

###  ✔ test Phase 1

In [1]:
from app.config import RSS_FEEDS
from app.collectors.rss import RSSCollector

for source_name, url in RSS_FEEDS.items():
    collector = RSSCollector(source_name, url)
    articles = collector.collect()
    print("source_name", source_name)
    print("articles", articles)
   
print(len(articles))

# for test just keep first 5 articles
articles = articles[:3]




source_name OpenAI
articles [Article(title='Previewing GPT-5.6 Sol: a next-generation model', source='OpenAI', score=0.0, summary_len=0, categories=None, content_len=162), Article(title='How agents are transforming work', source='OpenAI', score=0.0, summary_len=0, categories=None, content_len=147), Article(title='OpenAI and Broadcom unveil LLM-optimized inference chip', source='OpenAI', score=0.0, summary_len=0, categories=None, content_len=145), Article(title='Helping build shared standards for advanced AI', source='OpenAI', score=0.0, summary_len=0, categories=None, content_len=157), Article(title='How GPT-5 helped immunologist Derya Unutmaz solve a 3-year-old mystery', source='OpenAI', score=0.0, summary_len=0, categories=None, content_len=158), Article(title='How Omio is building the future of conversational travel', source='OpenAI', score=0.0, summary_len=0, categories=None, content_len=146), Article(title='Patch the Planet: a Daybreak initiative to support open source maintainers

In [2]:
articles

[Article(title='Previewing GPT-5.6 Sol: a next-generation model', source='OpenAI', score=0.0, summary_len=0, categories=None, content_len=162),
 Article(title='How agents are transforming work', source='OpenAI', score=0.0, summary_len=0, categories=None, content_len=147),
 Article(title='OpenAI and Broadcom unveil LLM-optimized inference chip', source='OpenAI', score=0.0, summary_len=0, categories=None, content_len=145)]

##  ✅ Phase 2 : AI Summarization
where the project starts becoming an AI Engineering project rather than just a data collection project.

- ✔ Build LLM Provider (llm/provider.py)
- ✔ Add Prompt Engineering
- ✔ Build Summarizer Service (services/summarizer.py)
- ✔ Update RSS Collector


###  ✔ test Phase 2

In [3]:
from app.collectors.rss import RSSCollector
from app.config import RSS_FEEDS
from app.services.summarizer import Summarizer

summarizer = Summarizer()

for article in articles:
    article = summarizer.summarize(article)

    print("=" * 80)
    print(article.title)
    print()
    print(article.url)
    print()
    print("article content:" , article.content)
    print()
    print("summary:" , article.summary)
    print()

Previewing GPT-5.6 Sol: a next-generation model

https://openai.com/index/previewing-gpt-5-6-sol

article content: OpenAI previews GPT-5.6 Sol, a next-generation model with stronger capabilities in coding, science, and cybersecurity, paired with its most advanced safety stack.

summary: Here's a concise summary of the article:

*   **Company/Model:** OpenAI introduced its next-generation model, GPT-5.6 Sol.
*   **Main Innovation:** This model features significantly stronger capabilities in coding, science, and cybersecurity, coupled with the most advanced safety stack to date.
*   **Key Capabilities:** Enhanced performance specifically in technical domains like coding, scientific reasoning, and cybersecurity tasks.
*   **Why it Matters:** The improved capabilities promise more powerful assistance in complex problem-solving, while the enhanced safety stack aims to mitigate risks associated with advanced AI systems.

How agents are transforming work

https://openai.com/index/how-agents-a

In [4]:
articles

[Article(title='Previewing GPT-5.6 Sol: a next-generation model', source='OpenAI', score=0.0, summary_len=639, categories=None, content_len=162),
 Article(title='How agents are transforming work', source='OpenAI', score=0.0, summary_len=416, categories=None, content_len=147),
 Article(title='OpenAI and Broadcom unveil LLM-optimized inference chip', source='OpenAI', score=0.0, summary_len=653, categories=None, content_len=145)]

##  ✅  Phase 3 : remove duplicates
```
Summaries
      │
      ▼
Embedding Model
      │
      ▼
Vectors
      │
      ▼
Cosine Similarity
      │
      ▼
Duplicate Detector
      │
      ▼
Unique Articles
```
on this Phase I will use :
- Sentence embeddings
- Vector representations
- Cosine similarity
- Similarity thresholds
- Building a reusable AI service
- Preparing for vector databases like Pinecone, Weaviate, Chroma, and FAISS

Project Structure

We'll add a new service.
```
app/

    services/
        summarizer.py
        deduplicator.py

    embeddings/
        encoder.py
```

###  ✔test Phase 3

In [5]:
from app.services.deduplicator import Deduplicator
deduplicator = Deduplicator()
unique_articles = deduplicator.remove_duplicates(articles)

print(f"Collected: {len(articles)}")
articles=unique_articles
print(f"Unique: {len(unique_articles)}")
unique_articles=[]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded successfully
Encoding 3 articles for deduplication...
Deduplication complete: 3 → 3 articles
Collected: 3
Unique: 3


##  ✅ Phase 4 : categorization
The next stage is AI-powered topic classification, where each unique article is assigned categories such as:
- LLMs
- Robotics
- Computer Vision
- AI Research
- Open Source
- Startups
- Healthcare AI
- Autonomous Vehicles

Step 4 Architecture
```
Articles (unique)
      │
      ▼
LLM Classifier OR Zero-shot Model
      │
      ▼
Structured Output (JSON categories)
      │
      ▼
Validated Article
```

steps :
1. Define Taxonomy (VERY IMPORTANT) (app/services/taxonomy.py)
2. Choose Classification Method 
    - Option A (Recommended): LLM-based structured classification
        - Flexible
        - High accuracy
        - Easy to iterate
    - Option B: Zero-shot transformer model
        - Faster
        - No API cost
        - Less accurate
3. Prompt Design  (app/prompts/classifier.py)

In [6]:
from app.services.classifier import Classifier
classifier = Classifier()
classified_articles = []

for article in articles:
    
    article = classifier.classify(article)

    print("title:", article.title)
    print("summary:", article.summary)
    print("categories:", article.categories)
    print("=" * 80)



title: Previewing GPT-5.6 Sol: a next-generation model
summary: Here's a concise summary of the article:

*   **Company/Model:** OpenAI introduced its next-generation model, GPT-5.6 Sol.
*   **Main Innovation:** This model features significantly stronger capabilities in coding, science, and cybersecurity, coupled with the most advanced safety stack to date.
*   **Key Capabilities:** Enhanced performance specifically in technical domains like coding, scientific reasoning, and cybersecurity tasks.
*   **Why it Matters:** The improved capabilities promise more powerful assistance in complex problem-solving, while the enhanced safety stack aims to mitigate risks associated with advanced AI systems.
categories: ['LLMs', 'Cybersecurity AI', 'Data Science']
title: How agents are transforming work
summary: Here's a concise summary of the article:

*   **Company/Organization:** OpenAI
*   **Main Innovation:** AI agents are shown to transform work by enabling longer, more complex tasks.
*   **Pr

In [7]:
articles

[Article(title='Previewing GPT-5.6 Sol: a next-generation model', source='OpenAI', score=0.0, summary_len=639, categories=['LLMs', 'Cybersecurity AI', 'Data Science'], content_len=162),
 Article(title='How agents are transforming work', source='OpenAI', score=0.0, summary_len=416, categories=['LLMs', 'AI Research', 'Data Science'], content_len=147),
 Article(title='OpenAI and Broadcom unveil LLM-optimized inference chip', source='OpenAI', score=0.0, summary_len=653, categories=['LLMs', 'AI Research'], content_len=145)]

##  ✅ Phase 5 : AI Ranking Agent
Architecture
```
Articles
   │
   ▼
Ranking Agent (LLM + heuristics)
   │
   ▼
Score per article (0–100)
   │
   ▼
Sorted Feed
```

ranking system :
1. Technical Impact (0–25)
2. Industry Importance (0–25)
3. Recency / Breaking News (0–20)
4. AI/ML Relevance (0–20)
5. Source Credibility (0–10)

steps : 
1. Extend Article Model
2. Ranking Prompt Design (app/prompts/ranker.py)
3. Ranking Agent Service (app/agents/news_agent.py)
4. Sorting Layer
5. Integrate into Pipeline

###  ✔ test Phase 5

In [8]:
from app.agents.news_agent import NewsRankingAgent
ranker = NewsRankingAgent()
processed = []
for article in articles:
    article = ranker.rank(article)

    print(article.title)
    print(article.score)
    print(article.score_breakdown)
    print("=" * 90)

Previewing GPT-5.6 Sol: a next-generation model
77
{'technical_impact': 20, 'industry_importance': 20, 'recency': 10, 'ai_relevance': 18, 'source_credibility': 9, 'total': 77}
How agents are transforming work
82
{'technical_impact': 20, 'industry_importance': 20, 'recency': 15, 'ai_relevance': 18, 'source_credibility': 9, 'total': 82}
OpenAI and Broadcom unveil LLM-optimized inference chip
70
{'technical_impact': 25, 'industry_importance': 25, 'recency': 0, 'ai_relevance': 20, 'source_credibility': 0, 'total': 70}


##  ✅Phase 6 — Store
Without a database:

You fetch news → process → lose everything

With PostgreSQL:

You keep all articles over time
You can query:
“what was published yesterday?”
“what topics are trending this week?”

PostgreSQL table
``` SQL
CREATE TABLE articles (
    id SERIAL PRIMARY KEY,
    title VARCHAR(500) NOT NULL,
    summary TEXT  NULL,
    url VARCHAR(500) NOT NULL,
    published_at TIMESTAMP  NULL,
    source VARCHAR(100) NOT NULL ,
    score INTEGER NOT NULL ,
    score_breakdown JSONB NOT NULL ,
    categories TEXT[] NOT NULL ,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
```



In [ ]:
from app.database.database import insert_articles
from app.config import DATABASE_URL
from sqlalchemy import create_engine

engine = create_engine(DATABASE_URL)
#insert_articles(engine, articles)s

##  ✅ Phase 7 : Generate and send a newsletter
```
Ranked Articles
      │
      ▼
Newsletter Builder
      │
      ▼
HTML / Text Formatter
      │
      ▼
Email Service (SMTP / API)
      │
      ▼
User Inbox
```

##  ✅ Phase 8 : Fast API
Goal: turn the  script into an API service.

until now we have :

python main.py

After this step I will have :
``` 
API SERVER
   ↓
/run-pipeline
   ↓
RSS → AI → Newsletter → Email
```

- run api ` uvicorn app.api.main:app --host 0.0.0.0 --port 8001` 


steps :
- create a new file called `app/services/pipeline.py`
- move the main logic from `app/main.py` to `app/services/pipeline.py`
- in `app/api/main.py` import the pipeline function and call it in the `/run-pipeline` endpoint
- test http://192.168.2.130:8001/run-pipeline
